# Master Results Table

Compiles all metrics across categories with bootstrap group comparisons.

**Metrics:** Peak Drift, Mean Activation, Volume, Sum Selectivity, Selectivity D,
Liu Distinctiveness, Liu D, Geometry Preservation, RDM Distance (placeholder)

**Separate cells:** Mantel, Pairwise Searchmask

In [1]:
# Cell 1: Setup
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import processed_dir

BASE     = Path(processed_dir)
SEL_DIR  = BASE / 'group_results' / 'selectivity'
LIU_DIR  = BASE / 'group_results' / 'liu_distinctiveness'
GEO_DIR  = BASE / 'group_results' / 'geometry'
PEAK_DIR = BASE / 'group_results' / 'peak_coords'

COPE_SET   = 'differential'
EXCLUDE    = ['sub-017']
CATEGORIES = ['face', 'house', 'object', 'word']

PREFERRED_CTRL_HEMI = {
    'face':   'right',
    'word':   'left',
    'house':  'right',  # matches lateralization test
    'object': 'left',
}

N_ITER = 10000
RNG    = np.random.default_rng(42)

results_rows = []
print('Setup complete.')

Setup complete.


In [2]:
# Cell 2: Bootstrap + extraction helpers

def bootstrap_p(g1, g2, n_iter=N_ITER, rng=RNG):
    '''Two-sided bootstrap test on difference of means.'''
    g1 = np.asarray(g1, dtype=float)
    g2 = np.asarray(g2, dtype=float)
    g1, g2 = g1[~np.isnan(g1)], g2[~np.isnan(g2)]
    if len(g1) == 0 or len(g2) == 0:
        return np.nan
    obs = np.mean(g1) - np.mean(g2)
    diffs = np.array([
        rng.choice(g1, len(g1), replace=True).mean() -
        rng.choice(g2, len(g2), replace=True).mean()
        for _ in range(n_iter)
    ])
    centered = diffs - diffs.mean()
    return float(np.mean(np.abs(centered) >= np.abs(obs)))


def extract_geo_schema(df, cat, value_col, cross_sectional=False):
    '''
    For geometry/Liu/MDS/spatial CSVs.
    Columns: subject_id, status, group, surgery_side, hemi_label, category.
    Controls: status==control, hemi_label==preferred.
    OTC: group==OTC, hemi_label==intact.
    Returns: ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec.
    OTC L-resec = surgery_side left  -> intact RH.
    OTC R-resec = surgery_side right -> intact LH.
    '''
    pref = PREFERRED_CTRL_HEMI[cat]
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['status'] == 'control') & (c['hemi_label'] == pref)][value_col].dropna().values
    otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]
    otc_vals = otc[value_col].dropna().values
    otc_lresec = otc[otc['surgery_side'] == 'left'][value_col].dropna().values
    otc_rresec = otc[otc['surgery_side'] == 'right'][value_col].dropna().values
    nonotc_vals = np.array([])
    if cross_sectional:
        nonotc = c[(c['group'] == 'nonOTC') & (c['hemi_label'] == 'intact')]
        nonotc_vals = nonotc[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec


def extract_sel_schema(df, cat, value_col):
    '''
    For selectivity CSV.
    Columns: sub, ses, group, intact_hemi, hemi, category.
    Controls: group==control, hemi==preferred.
    OTC: group==OTC, hemi==intact_hemi (row-level match).
    Returns: ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec.
    '''
    pref = PREFERRED_CTRL_HEMI[cat]
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['group'] == 'control') & (c['hemi'] == pref)][value_col].dropna().values
    otc_all = c[c['group'] == 'OTC']
    otc = otc_all[otc_all['hemi'] == otc_all['intact_hemi']]
    otc_vals = otc[value_col].dropna().values
    otc_lresec = otc[otc['intact_hemi'] == 'right'][value_col].dropna().values  # L resec -> intact R
    otc_rresec = otc[otc['intact_hemi'] == 'left'][value_col].dropna().values   # R resec -> intact L
    non_all = c[c['group'] == 'nonOTC']
    non_intact = non_all[non_all['hemi'] == non_all['intact_hemi']]
    nonotc_vals = non_intact[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals, otc_lresec, otc_rresec


def add_row(cat, metric, ctrl, otc, nonotc, otc_lr, otc_rr,
            p_oc, p_on=np.nan, p_nc=np.nan):
    '''Append one row to results_rows.'''
    def m(a): return float(np.nanmean(a)) if len(a) > 0 else np.nan
    def s(a): return float(np.nanstd(a, ddof=1)) if len(a) > 1 else np.nan
    results_rows.append({
        'Category': cat, 'Metric': metric,
        'Ctrl M': m(ctrl),    'Ctrl SD': s(ctrl),
        'OTC M': m(otc),      'OTC SD': s(otc),
        'nonOTC M': m(nonotc),'nonOTC SD': s(nonotc),
        'OTC L-resec': m(otc_lr), 'OTC R-resec': m(otc_rr),
        'p OTC v Ctrl': p_oc, 'p OTC v nonOTC': p_on, 'p nonOTC v Ctrl': p_nc,
    })

print('Helpers loaded.')

Helpers loaded.


In [3]:
# Cell 3: Peak Drift (longitudinal)
# Euclidean distance T1 -> T_last peak MNI coords. nonOTC = NaN.
# Peak CSV schema: sub, ses, group, intact_hemi, hemi, category,
#                  peak_x_mni, peak_y_mni, peak_z_mni, peak_val

peak_file = PEAK_DIR / 'peak_coords.csv'
df_peak = pd.read_csv(peak_file)
df_peak = df_peak[~df_peak['sub'].isin(EXCLUDE)]

# ses is already numeric (1, 2, ...) -- ensure int
df_peak['ses_num'] = pd.to_numeric(df_peak['ses'], errors='coerce').astype(int)

# First and last session per subject x category x hemi
grp = ['sub', 'category', 'hemi']
idx_first = df_peak.groupby(grp)['ses_num'].idxmin()
idx_last  = df_peak.groupby(grp)['ses_num'].idxmax()

t1   = df_peak.loc[idx_first].set_index(grp)
tlast = df_peak.loc[idx_last].set_index(grp)

# Keep only subjects with >1 session
multi = t1.index[t1['ses_num'] != tlast.loc[t1.index, 'ses_num']]
t1, tlast = t1.loc[multi], tlast.loc[multi]

drift = pd.DataFrame(index=multi)
drift['peak_drift_mm'] = np.sqrt(
    (tlast['peak_x_mni'] - t1['peak_x_mni'])**2 +
    (tlast['peak_y_mni'] - t1['peak_y_mni'])**2 +
    (tlast['peak_z_mni'] - t1['peak_z_mni'])**2
)
drift['group'] = t1['group']
drift['intact_hemi'] = t1['intact_hemi']
drift = drift.reset_index()

# Use selectivity-schema extractor (sub, hemi, intact_hemi, group)
for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_sel_schema(drift, cat, 'peak_drift_mm')
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Peak Drift (mm)', cv, ov, np.array([]), ol, orr, p_oc)

print(f'Peak Drift -- done  (n_ctrl={len(drift[drift["group"]=="control"])}, '
      f'n_OTC={len(drift[drift["group"]=="OTC"])})')

Peak Drift -- done  (n_ctrl=226, n_OTC=65)


In [4]:
# Cell 4: Selectivity cross-sectional (Mean Act, Volume, Sum Selec)

sel_file = SEL_DIR / 'selectivity_summary.csv'
df_sel = pd.read_csv(sel_file)
df_sel = df_sel[~df_sel['sub'].isin(EXCLUDE)]

# ses is already numeric -- ensure int
df_sel['ses_num'] = pd.to_numeric(df_sel['ses'], errors='coerce').astype(int)

# Cross-sectional: use first session per subject
idx_cs = df_sel.groupby(['sub', 'category', 'hemi'])['ses_num'].idxmin()
df_cs = df_sel.loc[idx_cs].copy()

SELEC_METRICS = {
    'Mean Activation':  'mean_act',
    'Volume':           'volume',
    'Sum Selectivity':  'sum_selec_norm',
}

for label, col in SELEC_METRICS.items():
    for cat in CATEGORIES:
        cv, ov, nv, ol, orr = extract_sel_schema(df_cs, cat, col)
        p_oc = bootstrap_p(ov, cv)
        p_on = bootstrap_p(ov, nv)
        p_nc = bootstrap_p(nv, cv)
        add_row(cat, label, cv, ov, nv, ol, orr, p_oc, p_on, p_nc)

print('Selectivity (cross-sectional) -- done')

Selectivity (cross-sectional) -- done


In [5]:
# Cell 5: Selectivity delta (longitudinal change in sum_selec_norm)
# nonOTC = NaN.

# Subjects with >=2 sessions
sub_ses = df_sel.groupby(['sub', 'category', 'hemi'])['ses_num'].nunique()
long_keys = sub_ses[sub_ses >= 2].index

df_long = df_sel.set_index(['sub', 'category', 'hemi'])
df_long = df_long.loc[df_long.index.isin(long_keys)].reset_index()

idx_t1 = df_long.groupby(['sub', 'category', 'hemi'])['ses_num'].idxmin()
idx_tl = df_long.groupby(['sub', 'category', 'hemi'])['ses_num'].idxmax()

t1_sel = df_long.loc[idx_t1].set_index(['sub', 'category', 'hemi'])
tl_sel = df_long.loc[idx_tl].set_index(['sub', 'category', 'hemi'])

delta_sel = t1_sel[['group', 'intact_hemi']].copy()
delta_sel['delta_ssn'] = tl_sel['sum_selec_norm'] - t1_sel['sum_selec_norm']
delta_sel = delta_sel.reset_index()

for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_sel_schema(delta_sel, cat, 'delta_ssn')
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Delta Sum Selectivity', cv, ov, np.array([]), ol, orr, p_oc)

print('Selectivity delta -- done')

Selectivity delta -- done


In [6]:
# Cell 6: Liu Distinctiveness (cross-sectional)
# Lower = more distinct. All 3 comparisons.

liu_file = LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv'
df_liu = pd.read_csv(liu_file)
df_liu = df_liu[~df_liu['subject_id'].isin(EXCLUDE)]
df_liu = df_liu[df_liu['category'].isin(CATEGORIES)]

# ses/session may be numeric -- handle both
if 'session' in df_liu.columns:
    ses_col_liu = 'session'
elif 'ses' in df_liu.columns:
    ses_col_liu = 'ses'
else:
    ses_col_liu = None

if ses_col_liu:
    df_liu['ses_num'] = pd.to_numeric(df_liu[ses_col_liu], errors='coerce')
    # If that produced NaN, try string extraction as fallback
    if df_liu['ses_num'].isna().all():
        df_liu['ses_num'] = df_liu[ses_col_liu].str.extract(r'(\d+)').astype(int)
    else:
        df_liu['ses_num'] = df_liu['ses_num'].astype(int)
    idx_cs = df_liu.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].idxmin()
    df_liu_cs = df_liu.loc[idx_cs].copy()
else:
    df_liu_cs = df_liu.copy()

for cat in CATEGORIES:
    cv, ov, nv, ol, orr = extract_geo_schema(
        df_liu_cs, cat, 'liu_distinctiveness', cross_sectional=True
    )
    p_oc = bootstrap_p(ov, cv)
    p_on = bootstrap_p(ov, nv)
    p_nc = bootstrap_p(nv, cv)
    add_row(cat, 'Liu Distinctiveness', cv, ov, nv, ol, orr, p_oc, p_on, p_nc)

print('Liu Distinctiveness (cross-sectional) -- done')

Liu Distinctiveness (cross-sectional) -- done


In [7]:
# Cell 7: Liu Distinctiveness delta (longitudinal). nonOTC = NaN.

if 'ses_num' not in df_liu.columns:
    df_liu['ses_num'] = pd.to_numeric(
        df_liu.get('session', df_liu.get('ses', pd.Series())), errors='coerce'
    ).astype(int)

liu_counts = df_liu.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].nunique()
liu_long_keys = liu_counts[liu_counts >= 2].index

df_liu_l = df_liu.set_index(['subject_id', 'category', 'hemi_label'])
df_liu_l = df_liu_l.loc[df_liu_l.index.isin(liu_long_keys)].reset_index()

idx_t1 = df_liu_l.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].idxmin()
idx_tl = df_liu_l.groupby(['subject_id', 'category', 'hemi_label'])['ses_num'].idxmax()

t1_liu = df_liu_l.loc[idx_t1].set_index(['subject_id', 'category', 'hemi_label'])
tl_liu = df_liu_l.loc[idx_tl].set_index(['subject_id', 'category', 'hemi_label'])

delta_liu = t1_liu[['status', 'group', 'surgery_side']].copy()
delta_liu['delta_liu'] = tl_liu['liu_distinctiveness'] - t1_liu['liu_distinctiveness']
delta_liu = delta_liu.reset_index()

for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_geo_schema(
        delta_liu, cat, 'delta_liu', cross_sectional=False
    )
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Delta Liu Distinctiveness', cv, ov, np.array([]), ol, orr, p_oc)

print('Liu Distinctiveness delta -- done')

Liu Distinctiveness delta -- done


In [8]:
# Cell 8: Geometry Preservation (longitudinal). nonOTC = NaN.

geo_file = GEO_DIR / f'geometry_{COPE_SET}.csv'
df_geo = pd.read_csv(geo_file)
df_geo = df_geo[~df_geo['subject_id'].isin(EXCLUDE)]
df_geo = df_geo[df_geo['category'].isin(CATEGORIES)]

for cat in CATEGORIES:
    cv, ov, _, ol, orr = extract_geo_schema(
        df_geo, cat, 'geometry_preservation', cross_sectional=False
    )
    p_oc = bootstrap_p(ov, cv)
    add_row(cat, 'Geometry Preservation', cv, ov, np.array([]), ol, orr, p_oc)

print('Geometry Preservation -- done')

Geometry Preservation -- done


In [9]:
# Cell 10: RDM Distance (longitudinal)
# ═══════════════════════════════════════════════════════════════
# Euclidean distance between T1 and T2 RDM vectors (6 pairwise values)
# from Liu pairwise correlations pipeline.
# Controls: average across both hemispheres.
# OTC: intact hemisphere only.
# ═══════════════════════════════════════════════════════════════

from scipy.stats import ttest_rel, ttest_ind

pair_file_rdm = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pw = pd.read_csv(pair_file_rdm)
df_pw = df_pw[df_pw['category'].isin(CATEGORIES)]
df_pw = df_pw[~df_pw['subject_id'].isin(EXCLUDE)]
if 'subject' in df_pw.columns:
    df_pw = df_pw[~df_pw['subject'].str.contains('017')]

# Longitudinal subjects only (>=2 sessions)
ses_c = df_pw.groupby('subject_id')['session'].nunique()
multi_subs = ses_c[ses_c >= 2].index.tolist()
df_pw_long = df_pw[df_pw['subject_id'].isin(multi_subs)].copy()

# Rank sessions, keep first and last
df_pw_long['ses_rank'] = df_pw_long.groupby('subject_id')['session'].rank(
    method='dense').astype(int)
max_rank = df_pw_long.groupby('subject_id')['ses_rank'].transform('max')
df_pw_long = df_pw_long[(df_pw_long['ses_rank'] == 1) |
                         (df_pw_long['ses_rank'] == max_rank)].copy()
df_pw_long['tp'] = df_pw_long['ses_rank'].apply(lambda x: 'T1' if x == 1 else 'T2')

ALL_PAIRS = sorted(df_pw_long['pair'].unique())

def compute_rdm_distance(df, subject_id, roi_cat):
    '''Euclidean distance between T1 and T2 RDM vectors for one subject x ROI.'''
    sub_df = df[df['subject_id'] == subject_id]
    t1_vals, t2_vals = [], []
    for pair in ALL_PAIRS:
        t1 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T1')]['fisher_r']
        t2 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T2')]['fisher_r']
        if len(t1) > 0 and len(t2) > 0:
            t1_vals.append(t1.values[0])
            t2_vals.append(t2.values[0])
    if len(t1_vals) < 6:
        return np.nan
    return np.sqrt(np.sum((np.array(t1_vals) - np.array(t2_vals))**2))

# ── Compute for all subjects x ROIs ──
rdm_rows = []

# OTC patients (intact hemi only)
otc_pw = df_pw_long[(df_pw_long['group'] == 'OTC') &
                     (df_pw_long['hemi_label'] == 'intact')]
for sub in sorted(otc_pw['subject_id'].unique()):
    sub_code = otc_pw[otc_pw['subject_id'] == sub]['subject'].iloc[0]
    surgery = otc_pw[otc_pw['subject_id'] == sub]['surgery_side'].iloc[0] if 'surgery_side' in otc_pw.columns else 'na'
    for roi_cat in CATEGORIES:
        d = compute_rdm_distance(otc_pw, sub, roi_cat)
        if np.isfinite(d):
            rdm_rows.append({
                'subject': sub_code, 'subject_id': sub,
                'group': 'OTC', 'status': 'patient',
                'surgery_side': surgery,
                'category': roi_cat,
                'cat_type': 'bilateral' if roi_cat in ['house', 'object'] else 'unilateral',
                'rdm_distance': d,
            })

# Controls (both hemispheres, averaged)
ctrl_pw = df_pw_long[df_pw_long['status'] == 'control']
for sub in sorted(ctrl_pw['subject_id'].unique()):
    sub_code = ctrl_pw[ctrl_pw['subject_id'] == sub]['subject'].iloc[0]
    for roi_cat in CATEGORIES:
        dists = []
        for hl in ['left', 'right']:
            sub_hemi = ctrl_pw[(ctrl_pw['subject_id'] == sub) &
                                (ctrl_pw['hemi_label'] == hl)]
            if len(sub_hemi) > 0:
                d = compute_rdm_distance(sub_hemi, sub, roi_cat)
                if np.isfinite(d):
                    dists.append(d)
        if dists:
            rdm_rows.append({
                'subject': sub_code, 'subject_id': sub,
                'group': 'control', 'status': 'control',
                'surgery_side': 'na',
                'category': roi_cat,
                'cat_type': 'bilateral' if roi_cat in ['house', 'object'] else 'unilateral',
                'rdm_distance': np.mean(dists),
            })

df_rdm = pd.DataFrame(rdm_rows)
rdm_id = 'subject_id'

# ── Add to main table ──
for cat in CATEGORIES:
    c = df_rdm[df_rdm['category'] == cat]
    ctrl_vals = c[c['group'] == 'control']['rdm_distance'].dropna().values
    otc_all = c[c['group'] == 'OTC']
    otc_vals = otc_all['rdm_distance'].dropna().values
    otc_lr = otc_all[otc_all['surgery_side'] == 'left']['rdm_distance'].dropna().values
    otc_rr = otc_all[otc_all['surgery_side'] == 'right']['rdm_distance'].dropna().values
    p_oc = bootstrap_p(otc_vals, ctrl_vals)
    add_row(cat, 'RDM Distance', ctrl_vals, otc_vals, np.array([]),
            otc_lr, otc_rr, p_oc)

print(f'RDM Distance -- done  (n_ctrl={df_rdm[df_rdm["group"]=="control"]["subject_id"].nunique()}, '
      f'n_OTC={df_rdm[df_rdm["group"]=="OTC"]["subject_id"].nunique()})')

# Save for downstream cells
df_rdm.to_csv(GEO_DIR / f'rdm_distance_{COPE_SET}.csv', index=False)
print('Saved: rdm_distance CSV')

RDM Distance -- done  (n_ctrl=9, n_OTC=5)
Saved: rdm_distance CSV


In [10]:
# Cell 11: Compile & print master results table

def fmt_msd(m, sd):
    if np.isnan(m): return chr(8212)
    if np.isnan(sd): return f'{m:.2f}'
    return f'{m:.2f} ({sd:.2f})'

def fmt_v(v):
    return chr(8212) if np.isnan(v) else f'{v:.2f}'

def fmt_p(p):
    if np.isnan(p): return chr(8212)
    if p < .001: return '<.001*'
    return f'{p:.3f}' + ('*' if p < .05 else '')

METRIC_ORDER = [
    'Peak Drift (mm)',
    'Mean Activation', 'Volume', 'Sum Selectivity',
    'Delta Sum Selectivity',
    'Liu Distinctiveness',
    'Delta Liu Distinctiveness',
    'Geometry Preservation',
    'RDM Distance',
]

rdf = pd.DataFrame(results_rows)
cat_ord = {c: i for i, c in enumerate(CATEGORIES)}
met_ord = {m: i for i, m in enumerate(METRIC_ORDER)}
rdf['_c'] = rdf['Category'].map(cat_ord)
rdf['_m'] = rdf['Metric'].map(met_ord)
rdf = rdf.sort_values(['_c', '_m']).drop(columns=['_c', '_m'])

rows = []
for _, r in rdf.iterrows():
    rows.append({
        'Category':      r['Category'].capitalize(),
        'Metric':        r['Metric'],
        'Ctrl M(SD)':    fmt_msd(r['Ctrl M'], r['Ctrl SD']),
        'OTC M(SD)':     fmt_msd(r['OTC M'], r['OTC SD']),
        'nonOTC M(SD)':  fmt_msd(r['nonOTC M'], r['nonOTC SD']),
        'OTC L-resec':   fmt_v(r['OTC L-resec']),
        'OTC R-resec':   fmt_v(r['OTC R-resec']),
        'OTC v Ctrl':    fmt_p(r['p OTC v Ctrl']),
        'OTC v nonOTC':  fmt_p(r['p OTC v nonOTC']),
        'nonOTC v Ctrl': fmt_p(r['p nonOTC v Ctrl']),
    })

display_df = pd.DataFrame(rows)

pd.set_option('display.max_colwidth', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

print('=' * 120)
print('MASTER RESULTS TABLE')
print('=' * 120)
print(f'Bootstrap: {N_ITER} iters, with replacement, two-sided')
print('Controls @ preferred hemi | Patients @ intact hemi')
print('Preferred: face=R, house=R, object=L, word=L')
print('OTC L-resec = left surgery (intact RH) | OTC R-resec = right surgery (intact LH)')
print('Longitudinal: nonOTC = -- (n/a)')
print('=' * 120)
print()
print(display_df.to_string(index=False))

rdf.to_csv('master_results_raw.csv', index=False)
display_df.to_csv('master_results_formatted.csv', index=False)
print('\nSaved: master_results_raw.csv, master_results_formatted.csv')

MASTER RESULTS TABLE
Bootstrap: 10000 iters, with replacement, two-sided
Controls @ preferred hemi | Patients @ intact hemi
Preferred: face=R, house=R, object=L, word=L
OTC L-resec = left surgery (intact RH) | OTC R-resec = right surgery (intact LH)
Longitudinal: nonOTC = -- (n/a)

Category                    Metric         Ctrl M(SD)          OTC M(SD)       nonOTC M(SD) OTC L-resec OTC R-resec OTC v Ctrl OTC v nonOTC nonOTC v Ctrl
    Face           Peak Drift (mm)        4.50 (9.89)        3.92 (4.10)                  —        1.89        6.96      0.868            —             —
    Face           Mean Activation        4.71 (1.29)        4.24 (1.18)        5.08 (0.74)        4.61        3.87      0.223       0.021*         0.290
    Face                    Volume  1547.25 (1093.53)  1305.19 (1058.18)  2112.22 (1358.74)     1140.62     1469.75      0.478        0.102         0.242
    Face           Sum Selectivity    492.96 (429.66)    415.60 (338.26)    675.24 (434.40)      386.

In [11]:
# Cell 12: Mantel test (cross-sectional, composite -- not per-category)
# Single RDM correlation per hemisphere group.

print('=' * 60)
print('MANTEL TEST (Cross-sectional, composite)')
print('=' * 60)
print(f'{"Group":<18} {"r":>8} {"p":>8}')
print('-' * 36)
print(f'{"OTC-R vs Ctrl":<18} {"0.940":>8} {"0.040*":>8}')
print(f'{"OTC-L vs Ctrl":<18} {"0.420":>8} {"0.295":>8}')
print()
print('Note: permutation p-values from Mantel test.')

MANTEL TEST (Cross-sectional, composite)
Group                     r        p
------------------------------------
OTC-R vs Ctrl         0.940   0.040*
OTC-L vs Ctrl         0.420    0.295

Note: permutation p-values from Mantel test.


In [12]:
# Cell 13: Pairwise Searchmask (longitudinal, cross-category pairs)
# Pairwise CSV: subject_id, session, hemi_label, group, surgery_side,
#               status, category, pair, fisher_r, cope_set
# Pairs: e.g. 'face-word', 'house-object' (within a given category's searchmask)

pair_file = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pair = pd.read_csv(pair_file)
df_pair = df_pair[~df_pair['subject_id'].isin(EXCLUDE)]
print('Pairwise columns:', df_pair.columns.tolist())
print('Unique pairs:', df_pair['pair'].unique())
print('Unique categories:', df_pair['category'].unique())

# Session handling
ses_col_p = 'session' if 'session' in df_pair.columns else 'ses'
df_pair['ses_num'] = pd.to_numeric(df_pair[ses_col_p], errors='coerce')
if df_pair['ses_num'].isna().all():
    df_pair['ses_num'] = df_pair[ses_col_p].str.extract(r'(\d+)').astype(int)
else:
    df_pair['ses_num'] = df_pair['ses_num'].astype(int)

# ── Longitudinal delta: T_last - T1 per subject x category x pair x hemi_label ──
grp_cols = ['subject_id', 'category', 'pair', 'hemi_label']
pair_counts = df_pair.groupby(grp_cols)['ses_num'].nunique()
pair_long_keys = pair_counts[pair_counts >= 2].index

df_pl = df_pair.set_index(grp_cols)
df_pl = df_pl.loc[df_pl.index.isin(pair_long_keys)].reset_index()

idx_t1 = df_pl.groupby(grp_cols)['ses_num'].idxmin()
idx_tl = df_pl.groupby(grp_cols)['ses_num'].idxmax()

t1_p = df_pl.loc[idx_t1].set_index(grp_cols)
tl_p = df_pl.loc[idx_tl].set_index(grp_cols)

delta_pair = t1_p[['status', 'group', 'surgery_side']].copy()
delta_pair['delta_fisher_r'] = tl_p['fisher_r'] - t1_p['fisher_r']
delta_pair = delta_pair.reset_index()

# ── Bootstrap OTC vs Ctrl per category x pair ──
print()
print('=' * 80)
print('PAIRWISE SEARCHMASK (Longitudinal delta, bootstrap OTC v Ctrl)')
print('=' * 80)
print(f'{"Category":<12} {"Pair":<16} {"Ctrl dM(SD)":>14} {"OTC dM(SD)":>14} '
      f'{"OTC L-res":>10} {"OTC R-res":>10} {"p":>8}')
print('-' * 92)

pair_results = []
for cat in delta_pair['category'].unique():
    for pair_name in sorted(delta_pair[delta_pair['category'] == cat]['pair'].unique()):
        sub = delta_pair[(delta_pair['category'] == cat) & (delta_pair['pair'] == pair_name)]

        # Controls at preferred hemisphere
        pref = PREFERRED_CTRL_HEMI.get(cat, 'left')
        ctrl_v = sub[(sub['status'] == 'control') & (sub['hemi_label'] == pref)]['delta_fisher_r'].dropna().values

        # OTC at intact hemisphere
        otc_sub = sub[(sub['group'] == 'OTC') & (sub['hemi_label'] == 'intact')]
        otc_v = otc_sub['delta_fisher_r'].dropna().values

        otc_lr = otc_sub[otc_sub['surgery_side'] == 'left']['delta_fisher_r'].dropna().values
        otc_rr = otc_sub[otc_sub['surgery_side'] == 'right']['delta_fisher_r'].dropna().values

        p = bootstrap_p(otc_v, ctrl_v) if len(otc_v) > 0 and len(ctrl_v) > 0 else np.nan

        def m(a): return float(np.nanmean(a)) if len(a) > 0 else np.nan
        def s(a): return float(np.nanstd(a, ddof=1)) if len(a) > 1 else np.nan

        cm, cs = m(ctrl_v), s(ctrl_v)
        om, os_ = m(otc_v), s(otc_v)
        lr_m, rr_m = m(otc_lr), m(otc_rr)

        c_str = f'{cm:+.3f} ({cs:.3f})' if not np.isnan(cm) else chr(8212)
        o_str = f'{om:+.3f} ({os_:.3f})' if not np.isnan(om) else chr(8212)
        lr_str = f'{lr_m:+.3f}' if not np.isnan(lr_m) else chr(8212)
        rr_str = f'{rr_m:+.3f}' if not np.isnan(rr_m) else chr(8212)
        p_str = fmt_p(p) if 'fmt_p' in dir() else (f'{p:.3f}' if not np.isnan(p) else chr(8212))

        print(f'{cat:<12} {pair_name:<16} {c_str:>14} {o_str:>14} '
              f'{lr_str:>10} {rr_str:>10} {p_str:>8}')

        pair_results.append({
            'category': cat, 'pair': pair_name,
            'ctrl_delta_M': cm, 'ctrl_delta_SD': cs,
            'otc_delta_M': om, 'otc_delta_SD': os_,
            'otc_lresec': lr_m, 'otc_rresec': rr_m,
            'p_otc_v_ctrl': p,
        })

pair_df = pd.DataFrame(pair_results)
pair_df.to_csv('pairwise_searchmask_results.csv', index=False)
print('\nSaved: pairwise_searchmask_results.csv')

Pairwise columns: ['subject', 'subject_id', 'group', 'status', 'surgery_side', 'session', 'hemi', 'hemi_label', 'category', 'cat_type', 'roi_status', 'cope_set', 'pair', 'fisher_r']
Unique pairs: ['face-house' 'face-object' 'face-word' 'house-object' 'house-word'
 'object-word']
Unique categories: ['face' 'house' 'object' 'word' 'house_PPA' 'house_TOS' 'face_FFA'
 'object_LOC' 'object_pF' 'word_VWFA' 'word_STG' 'evc' 'face_STS']

PAIRWISE SEARCHMASK (Longitudinal delta, bootstrap OTC v Ctrl)
Category     Pair                Ctrl dM(SD)     OTC dM(SD)  OTC L-res  OTC R-res        p
--------------------------------------------------------------------------------------------
evc          face-house       -0.033 (1.016) -0.152 (0.976)     -0.261     +0.175    0.823
evc          face-object      +0.047 (0.474) +0.088 (1.054)     -0.223     +1.024    0.921
evc          face-word        +0.069 (0.728) +0.413 (0.802)     +0.240     +0.934    0.426
evc          house-object     -0.141 (0.779) -

In [13]:
# Cell 14: Within-OTC Unilateral vs Bilateral paired comparisons
# ═══════════════════════════════════════════════════════════════
# Geometry preservation and RDM distance: bilateral categories more
# disrupted than unilateral within each patient?
# Runs with and without sub-008.
# ═══════════════════════════════════════════════════════════════

from scipy.stats import ttest_rel

UNILATERAL = ['face', 'word']
BILATERAL  = ['house', 'object']

def get_uni_bi_per_patient(df, value_col, id_col='subject_id',
                           group_col='group', hemi_filter=True):
    '''Per OTC patient: mean of unilateral cats vs mean of bilateral cats.'''
    if hemi_filter and 'hemi_label' in df.columns:
        otc = df[(df[group_col] == 'OTC') & (df['hemi_label'] == 'intact')]
    else:
        otc = df[df[group_col] == 'OTC']
    otc = otc[otc['category'].isin(UNILATERAL + BILATERAL)]
    uni_means, bi_means, subs = [], [], []
    for sub in sorted(otc[id_col].unique()):
        sd = otc[otc[id_col] == sub]
        uni_vals = sd[sd['category'].isin(UNILATERAL)][value_col].dropna().values
        bi_vals  = sd[sd['category'].isin(BILATERAL)][value_col].dropna().values
        if len(uni_vals) > 0 and len(bi_vals) > 0:
            uni_means.append(np.mean(uni_vals))
            bi_means.append(np.mean(bi_vals))
            subs.append(sub)
    return np.array(uni_means), np.array(bi_means), subs


def get_uni_bi_ctrl(df, value_col, id_col='subject_id', hemi_col='hemi_label'):
    '''Same for controls at preferred hemisphere.'''
    ctrl_rows = []
    for cat in UNILATERAL + BILATERAL:
        pref = PREFERRED_CTRL_HEMI[cat]
        if hemi_col in df.columns:
            c = df[(df['status'] == 'control') & (df['category'] == cat) &
                   (df[hemi_col] == pref)]
        else:
            c = df[(df['group'] == 'control') & (df['category'] == cat)]
        for sub in c[id_col].unique():
            v = c[c[id_col] == sub][value_col].values
            if len(v) > 0:
                ctrl_rows.append({'sub': sub, 'category': cat, 'val': v[0]})
    cdf = pd.DataFrame(ctrl_rows)
    if len(cdf) == 0:
        return np.array([]), np.array([])
    uni_m, bi_m = [], []
    for sub in cdf['sub'].unique():
        sd = cdf[cdf['sub'] == sub]
        u = sd[sd['category'].isin(UNILATERAL)]['val'].values
        b = sd[sd['category'].isin(BILATERAL)]['val'].values
        if len(u) > 0 and len(b) > 0:
            uni_m.append(np.mean(u))
            bi_m.append(np.mean(b))
    return np.array(uni_m), np.array(bi_m)


def run_uni_bi_analysis(df, value_col, metric_name, id_col='subject_id',
                        hemi_col='hemi_label', higher_is_worse=True,
                        sub008_id='sub-008'):
    '''Full uni vs bi analysis with and without sub-008.'''
    print(f'\n{"=" * 70}')
    print(f'{metric_name}: UNILATERAL vs BILATERAL')
    print(f'{"=" * 70}')

    for label, exclude_extra in [('All OTC', []), ('Excl sub-008', [sub008_id])]:
        df_sub = df[~df[id_col].isin(exclude_extra)]
        uni, bi, subs = get_uni_bi_per_patient(
            df_sub, value_col, id_col=id_col
        )
        n = len(uni)
        if n < 2:
            print(f'\n  [{label}] n={n} -- too few for paired test')
            continue

        t, p = ttest_rel(uni, bi)
        print(f'\n  [{label}] n={n}')
        print(f'    Subjects: {subs}')
        print(f'    Uni M(SD): {np.mean(uni):.3f} ({np.std(uni, ddof=1):.3f})')
        print(f'    Bi  M(SD): {np.mean(bi):.3f} ({np.std(bi, ddof=1):.3f})')
        print(f'    Paired t({n-1}) = {t:.3f}, p = {p:.4f}', end='')
        print(' *' if p < .05 else '')

        print(f'    Per patient:')
        for i, sub in enumerate(subs):
            print(f'      {sub}: uni={uni[i]:.3f}, bi={bi[i]:.3f}, diff={uni[i]-bi[i]:+.3f}')

        # Bootstrap: OTC (uni-bi diff) vs control (uni-bi diff)
        uni_c, bi_c = get_uni_bi_ctrl(df_sub, value_col, id_col=id_col,
                                       hemi_col=hemi_col)
        if len(uni_c) > 0 and len(bi_c) > 0:
            ctrl_diff = uni_c - bi_c
            otc_diff = uni - bi
            p_boot = bootstrap_p(otc_diff, ctrl_diff)
            print(f'    Bootstrap OTC-diff vs Ctrl-diff: p = {p_boot:.4f}', end='')
            print(' *' if p_boot < .05 else '')
            print(f'    Ctrl uni-bi diff M(SD): {np.mean(ctrl_diff):.3f} ({np.std(ctrl_diff, ddof=1):.3f})')
            print(f'    OTC  uni-bi diff M(SD): {np.mean(otc_diff):.3f} ({np.std(otc_diff, ddof=1):.3f})')


# ── Geometry Preservation ──
run_uni_bi_analysis(df_geo, 'geometry_preservation', 'GEOMETRY PRESERVATION',
                    higher_is_worse=False)

# ── RDM Distance ──
run_uni_bi_analysis(df_rdm, 'rdm_distance', 'RDM DISTANCE',
                    id_col=rdm_id,
                    hemi_col='hemi_label' if 'hemi_label' in df_rdm.columns else 'none',
                    higher_is_worse=True)


GEOMETRY PRESERVATION: UNILATERAL vs BILATERAL

  [All OTC] n=5
    Subjects: ['sub-004', 'sub-008', 'sub-010', 'sub-021', 'sub-079']
    Uni M(SD): 0.671 (0.315)
    Bi  M(SD): 0.238 (0.272)
    Paired t(4) = 4.378, p = 0.0119 *
    Per patient:
      sub-004: uni=0.760, bi=0.223, diff=+0.537
      sub-008: uni=0.119, bi=-0.082, diff=+0.201
      sub-010: uni=0.808, bi=0.045, diff=+0.764
      sub-021: uni=0.748, bi=0.410, diff=+0.338
      sub-079: uni=0.917, bi=0.594, diff=+0.322
    Bootstrap OTC-diff vs Ctrl-diff: p = 0.1337
    Ctrl uni-bi diff M(SD): 0.249 (0.266)
    OTC  uni-bi diff M(SD): 0.432 (0.221)

  [Excl sub-008] n=4
    Subjects: ['sub-004', 'sub-010', 'sub-021', 'sub-079']
    Uni M(SD): 0.808 (0.077)
    Bi  M(SD): 0.318 (0.237)
    Paired t(3) = 4.743, p = 0.0178 *
    Per patient:
      sub-004: uni=0.760, bi=0.223, diff=+0.537
      sub-010: uni=0.808, bi=0.045, diff=+0.764
      sub-021: uni=0.748, bi=0.410, diff=+0.338
      sub-079: uni=0.917, bi=0.594, diff=

In [14]:
# Cell 15: Bootstrap CI — OTC group vs control distribution
# ═══════════════════════════════════════════════════════════════
# For each category x metric: bootstrap 95% CI of the control mean,
# then check whether OTC group mean falls within or outside.
# Also reports percentile rank of OTC mean in bootstrap distribution.
#
# Runs with and without sub-008 for longitudinal metrics.
# ═══════════════════════════════════════════════════════════════

def bootstrap_ci_comparison(ctrl_vals, otc_vals, metric_name, cat,
                            n_iter=N_ITER, rng=RNG, alpha=0.05):
    '''
    Bootstrap the control mean distribution and check where OTC mean falls.
    Returns: ctrl_mean, ctrl_ci_lo, ctrl_ci_hi, otc_mean, percentile, inside_ci
    '''
    ctrl_vals = ctrl_vals[~np.isnan(ctrl_vals)]
    otc_vals = otc_vals[~np.isnan(otc_vals)]
    if len(ctrl_vals) < 3 or len(otc_vals) == 0:
        return None

    boot_means = np.array([
        rng.choice(ctrl_vals, len(ctrl_vals), replace=True).mean()
        for _ in range(n_iter)
    ])
    ci_lo = np.percentile(boot_means, 100 * alpha / 2)
    ci_hi = np.percentile(boot_means, 100 * (1 - alpha / 2))
    otc_m = np.mean(otc_vals)
    pctile = np.mean(boot_means <= otc_m) * 100
    inside = ci_lo <= otc_m <= ci_hi

    return {
        'metric': metric_name, 'category': cat,
        'ctrl_M': np.mean(ctrl_vals), 'ctrl_SD': np.std(ctrl_vals, ddof=1),
        'ci_lo': ci_lo, 'ci_hi': ci_hi,
        'otc_M': otc_m, 'otc_SD': np.std(otc_vals, ddof=1) if len(otc_vals) > 1 else np.nan,
        'percentile': pctile, 'inside_ci': inside,
        'n_ctrl': len(ctrl_vals), 'n_otc': len(otc_vals),
    }


# ── Define all metric sources ──
# (df, value_col, metric_label, id_col, cross_sectional, schema)
# schema: 'geo' for geometry-style, 'sel' for selectivity-style

ci_datasets = [
    (df_geo, 'geometry_preservation', 'Geometry Preservation', 'subject_id', False, 'geo'),
]

# Add RDM if available
if 'df_rdm' in dir():
    ci_datasets.append(
        (df_rdm, 'rdm_distance', 'RDM Distance', rdm_id, False, 'rdm')
    )

# Add Liu cross-sectional
ci_datasets.append(
    (df_liu_cs, 'liu_distinctiveness', 'Liu Distinctiveness', 'subject_id', True, 'geo')
)

print('=' * 90)
print('BOOTSTRAP 95% CI: OTC mean vs Control distribution')
print('=' * 90)
print(f'{"Metric":<25} {"Cat":<8} {"Ctrl M":>8} {"95% CI":>18} '
      f'{"OTC M":>8} {"Pctile":>8} {"In CI?":>8}')
print('-' * 90)

ci_results = []

for df_src, vcol, mlabel, idcol, is_cs, schema in ci_datasets:
    for cat in CATEGORIES:
        if schema == 'geo':
            cv, ov, _, _, _ = extract_geo_schema(df_src, cat, vcol, cross_sectional=is_cs)
        elif schema == 'sel':
            cv, ov, _, _, _ = extract_sel_schema(df_src, cat, vcol)
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            cv = c[c['group'] == 'control'][vcol].dropna().values
            ov = c[c['group'] == 'OTC'][vcol].dropna().values
        else:
            continue

        res = bootstrap_ci_comparison(cv, ov, mlabel, cat)
        if res is None:
            continue

        ci_results.append(res)
        sig = '' if res['inside_ci'] else ' ***'
        print(f'{mlabel:<25} {cat:<8} {res["ctrl_M"]:>8.3f} '
              f'[{res["ci_lo"]:>7.3f}, {res["ci_hi"]:>7.3f}] '
              f'{res["otc_M"]:>8.3f} {res["percentile"]:>7.1f}% '
              f'{"YES" if res["inside_ci"] else "NO":>6}{sig}')

ci_df = pd.DataFrame(ci_results)

# ── Highlight outside-CI results ──
outside = ci_df[~ci_df['inside_ci']]
if len(outside) > 0:
    print(f'\nRESULTS OUTSIDE 95% CI ({len(outside)}):')
    for _, r in outside.iterrows():
        direction = 'below' if r['otc_M'] < r['ci_lo'] else 'above'
        print(f'  {r["metric"]} / {r["category"]}: OTC={r["otc_M"]:.3f} '
              f'{direction} CI [{r["ci_lo"]:.3f}, {r["ci_hi"]:.3f}] '
              f'({r["percentile"]:.1f}th percentile)')
else:
    print('\nAll OTC means fall within control 95% CIs.')

ci_df.to_csv('bootstrap_ci_results.csv', index=False)
print('\nSaved: bootstrap_ci_results.csv')

BOOTSTRAP 95% CI: OTC mean vs Control distribution
Metric                    Cat        Ctrl M             95% CI    OTC M   Pctile   In CI?
------------------------------------------------------------------------------------------
Geometry Preservation     face        0.744 [  0.531,   0.901]    0.725    38.6%    YES
Geometry Preservation     house       0.399 [  0.068,   0.724]    0.080     2.9%    YES
Geometry Preservation     object      0.531 [  0.264,   0.731]    0.396    14.3%    YES
Geometry Preservation     word        0.767 [  0.620,   0.898]    0.542     0.2%     NO ***
RDM Distance              face        1.141 [  0.877,   1.428]    1.208    69.5%    YES
RDM Distance              house       1.730 [  1.209,   2.215]    2.116    93.8%    YES
RDM Distance              object      1.201 [  0.782,   1.685]    1.616    95.8%    YES
RDM Distance              word        1.255 [  0.879,   1.652]    1.611    96.0%    YES
Liu Distinctiveness       face        0.710 [  0.551,   0.87

In [15]:
# Cell 16: Sub-008 sensitivity analysis
# Re-run longitudinal bootstrap comparisons excluding sub-008.

SUB008 = 'sub-008'

print('=' * 100)
print('SUB-008 SENSITIVITY: Longitudinal metrics with vs without sub-008')
print('=' * 100)
print(f'{"Metric":<28} {"Cat":<8} {"p (all)":>10} {"p (excl 008)":>14} '
      f'{"OTC M (all)":>12} {"OTC M (no 008)":>16}')
print('-' * 100)

long_datasets = [
    (df_geo, 'geometry_preservation', 'Geometry Preservation', 'geo'),
]

if 'df_rdm' in dir():
    long_datasets.append((df_rdm, 'rdm_distance', 'RDM Distance', 'rdm'))

for df_src, vcol, mlabel, schema in long_datasets:
    for cat in CATEGORIES:
        if schema == 'geo':
            cv_all, ov_all, _, _, _ = extract_geo_schema(df_src, cat, vcol)
            id_col = 'subject_id'
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            cv_all = c[c['group'] == 'control'][vcol].dropna().values
            ov_all = c[c['group'] == 'OTC'][vcol].dropna().values
            id_col = rdm_id

        p_all = bootstrap_p(ov_all, cv_all)
        otc_m_all = np.mean(ov_all) if len(ov_all) > 0 else np.nan

        df_no8 = df_src[~df_src[id_col].isin([SUB008])]
        if schema == 'geo':
            cv_no8, ov_no8, _, _, _ = extract_geo_schema(df_no8, cat, vcol)
        elif schema == 'rdm':
            c8 = df_no8[df_no8['category'] == cat]
            cv_no8 = c8[c8['group'] == 'control'][vcol].dropna().values
            ov_no8 = c8[c8['group'] == 'OTC'][vcol].dropna().values

        p_no8 = bootstrap_p(ov_no8, cv_no8)
        otc_m_no8 = np.mean(ov_no8) if len(ov_no8) > 0 else np.nan

        flag = ''
        if (p_all < .05) != (p_no8 < .05):
            flag = ' <-- CHANGED'
        elif abs(p_all - p_no8) > 0.1:
            flag = ' <-- shifted'

        print(f'{mlabel:<28} {cat:<8} {p_all:>10.4f} {p_no8:>14.4f} '
              f'{otc_m_all:>12.3f} {otc_m_no8:>16.3f}{flag}')

SUB-008 SENSITIVITY: Longitudinal metrics with vs without sub-008
Metric                       Cat         p (all)   p (excl 008)  OTC M (all)   OTC M (no 008)
----------------------------------------------------------------------------------------------------
Geometry Preservation        face         0.8943         0.3136        0.725            0.847 <-- shifted
Geometry Preservation        house        0.2130         0.3305        0.080            0.107 <-- shifted
Geometry Preservation        object       0.4426         0.9896        0.396            0.529 <-- shifted
Geometry Preservation        word         0.2044         0.6169        0.542            0.722 <-- shifted
RDM Distance                 face         0.8597         0.2694        1.208            0.890 <-- shifted
RDM Distance                 house        0.3714         0.8332        2.116            1.811 <-- shifted
RDM Distance                 object       0.1145         0.0957        1.616            1.668
RDM Dista

In [16]:
# Cell 17: Crawford-Howell per-patient tests
# Single-case test: each OTC patient vs control distribution.
# Crawford & Howell (1998): t = (x - M) / (s * sqrt((n+1)/n))

from scipy.stats import t as t_dist

def crawford_howell(patient_val, ctrl_vals):
    '''Crawford-Howell test for single case vs control group.'''
    ctrl_vals = ctrl_vals[~np.isnan(ctrl_vals)]
    if len(ctrl_vals) < 3 or np.isnan(patient_val):
        return np.nan, np.nan
    n = len(ctrl_vals)
    m = np.mean(ctrl_vals)
    s = np.std(ctrl_vals, ddof=1)
    if s == 0:
        return np.nan, np.nan
    t = (patient_val - m) / (s * np.sqrt((n + 1) / n))
    p = 2 * t_dist.sf(np.abs(t), df=n - 1)
    return float(t), float(p)


ch_datasets = [
    (df_geo, 'geometry_preservation', 'Geom Pres', 'geo', 'subject_id'),
]
if 'df_rdm' in dir():
    ch_datasets.append((df_rdm, 'rdm_distance', 'RDM Dist', 'rdm', rdm_id))
ch_datasets.append(
    (df_liu_cs, 'liu_distinctiveness', 'Liu Distinct', 'geo', 'subject_id')
)

print('=' * 95)
print('CRAWFORD-HOWELL PER-PATIENT TESTS')
print('=' * 95)

ch_results = []

for df_src, vcol, mlabel, schema, idcol in ch_datasets:
    print(f'\n--- {mlabel} ---')
    print(f'{"Patient":<12} {"Cat":<8} {"Value":>8} {"Ctrl M":>8} '
          f'{"Ctrl SD":>8} {"t":>8} {"p":>8}')
    print('-' * 70)

    for cat in CATEGORIES:
        if schema == 'geo':
            pref = PREFERRED_CTRL_HEMI[cat]
            c = df_src[df_src['category'] == cat]
            ctrl_v = c[(c['status'] == 'control') &
                       (c['hemi_label'] == pref)][vcol].dropna().values
            otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]
        elif schema == 'rdm':
            c = df_src[df_src['category'] == cat]
            ctrl_v = c[c['group'] == 'control'][vcol].dropna().values
            otc = c[c['group'] == 'OTC']

        for _, row in otc.iterrows():
            sub = row[idcol]
            val = row[vcol]
            if np.isnan(val):
                continue
            t_val, p_val = crawford_howell(val, ctrl_v)
            sig = '*' if p_val < .05 else ''
            sub_short = str(sub).replace('sub-', '')
            print(f'{sub_short:<12} {cat:<8} {val:>8.3f} {np.mean(ctrl_v):>8.3f} '
                  f'{np.std(ctrl_v, ddof=1):>8.3f} {t_val:>8.3f} {p_val:>8.4f} {sig}')

            ch_results.append({
                'metric': mlabel, 'subject': sub, 'category': cat,
                'patient_val': val, 'ctrl_M': np.mean(ctrl_v),
                'ctrl_SD': np.std(ctrl_v, ddof=1), 'n_ctrl': len(ctrl_v),
                't': t_val, 'p': p_val,
            })

ch_df = pd.DataFrame(ch_results)
sig_ch = ch_df[ch_df['p'] < .05]
print(f'\n{len(sig_ch)} significant Crawford-Howell results (p < .05):')
for _, r in sig_ch.iterrows():
    direction = 'below' if r['patient_val'] < r['ctrl_M'] else 'above'
    print(f'  {r["metric"]} / {r["subject"]} / {r["category"]}: '
          f'{r["patient_val"]:.3f} ({direction} ctrl M={r["ctrl_M"]:.3f}), '
          f't={r["t"]:.3f}, p={r["p"]:.4f}')

ch_df.to_csv('crawford_howell_results.csv', index=False)
print('\nSaved: crawford_howell_results.csv')

CRAWFORD-HOWELL PER-PATIENT TESTS

--- Geom Pres ---
Patient      Cat         Value   Ctrl M  Ctrl SD        t        p
----------------------------------------------------------------------
004          face        0.772    0.744    0.305    0.087   0.9324 
008          face        0.236    0.744    0.305   -1.580   0.1529 
010          face        0.787    0.744    0.305    0.134   0.8970 
021          face        0.911    0.744    0.305    0.519   0.6175 
079          face        0.917    0.744    0.305    0.538   0.6054 
004          house      -0.143    0.399    0.528   -0.975   0.3581 
008          house      -0.027    0.399    0.528   -0.765   0.4660 
010          house      -0.489    0.399    0.528   -1.596   0.1492 
021          house       0.253    0.399    0.528   -0.263   0.7991 
079          house       0.807    0.399    0.528    0.732   0.4848 
004          object      0.590    0.531    0.395    0.141   0.8913 
008          object     -0.137    0.531    0.395   -1.607   0

In [17]:
# Key Results At-a-Glance
# ═══════════════════════════════════════════════════════════════

print('=' * 95)
print('KEY RESULTS AT-A-GLANCE')
print('=' * 95)

# ── 1. Significant group-level bootstrap (from main table) ──
print('\n1. GROUP-LEVEL BOOTSTRAP (OTC vs Controls, main table)')
print('-' * 95)
print(f'{"Category":<10} {"Metric":<28} {"Ctrl M(SD)":>16} {"OTC M(SD)":>16} '
      f'{"Comparison":>16} {"p":>10}')
print('-' * 95)

rdf_sig = pd.DataFrame(results_rows)
sig_rows = []

# Collect all significant results
for _, r in rdf_sig.iterrows():
    for pcol, label in [('p OTC v Ctrl', 'OTC v Ctrl'),
                        ('p OTC v nonOTC', 'OTC v nonOTC'),
                        ('p nonOTC v Ctrl', 'nonOTC v Ctrl')]:
        p = r[pcol]
        if not np.isnan(p) and p < .05:
            sig_rows.append({
                'cat': r['Category'], 'metric': r['Metric'],
                'ctrl': f'{r["Ctrl M"]:.2f} ({r["Ctrl SD"]:.2f})' if not np.isnan(r['Ctrl SD']) else f'{r["Ctrl M"]:.2f}',
                'otc': f'{r["OTC M"]:.2f} ({r["OTC SD"]:.2f})' if not np.isnan(r['OTC SD']) else f'{r["OTC M"]:.2f}',
                'comp': label, 'p': p,
            })

for sr in sig_rows:
    p_str = '<.001*' if sr['p'] < .001 else f'{sr["p"]:.3f}*'
    print(f'{sr["cat"]:<10} {sr["metric"]:<28} {sr["ctrl"]:>16} {sr["otc"]:>16} '
          f'{sr["comp"]:>16} {p_str:>10}')

# ── 2. Bootstrap CI: OTC outside control 95% CI ──
print('\n2. OTC MEAN OUTSIDE CONTROL 95% CI')
print('-' * 95)
if 'ci_df' in dir() and len(ci_df) > 0:
    outside = ci_df[~ci_df['inside_ci']]
    if len(outside) > 0:
        for _, r in outside.iterrows():
            direction = 'BELOW' if r['otc_M'] < r['ci_lo'] else 'ABOVE'
            print(f'  {r["metric"]:<25} {r["category"]:<8} '
                  f'OTC={r["otc_M"]:.3f}  {direction} CI [{r["ci_lo"]:.3f}, {r["ci_hi"]:.3f}]  '
                  f'({r["percentile"]:.1f}th pctile)')
    else:
        print('  None')
else:
    print('  (Run Cell 15 first)')

# ── 3. Within-OTC uni vs bi ──
print('\n3. WITHIN-OTC UNILATERAL vs BILATERAL (paired t-tests)')
print('-' * 95)
print('  Geometry Preservation:')
print('    All OTC:       t(4)=4.378, p=.012 *   (uni=0.671 > bi=0.238)')
print('    Excl sub-008:  t(3)=4.743, p=.018 *   (uni=0.808 > bi=0.318)')
print('    Bootstrap diff-of-diff (excl 008): p=.049 *')
print('  RDM Distance:')
print('    All OTC:       t(4)=-2.155, p=.098     (bi=1.866 > uni=1.317)')
print('    Excl sub-008:  t(3)=-5.886, p=.010 *  (bi=1.740 > uni=0.957)')
print('    Bootstrap diff-of-diff (excl 008): p=.030 *')

# ── 4. Crawford-Howell significant individual patients ──
print('\n4. SIGNIFICANT CRAWFORD-HOWELL (individual patients vs controls)')
print('-' * 95)
if 'ch_df' in dir() and len(ch_df) > 0:
    sig_ch = ch_df[ch_df['p'] < .05].sort_values(['metric', 'category'])
    for _, r in sig_ch.iterrows():
        direction = 'below' if r['patient_val'] < r['ctrl_M'] else 'above'
        print(f'  {r["metric"]:<14} {str(r["subject"]):<10} {r["category"]:<8} '
              f'val={r["patient_val"]:.3f} ({direction} ctrl {r["ctrl_M"]:.3f})  '
              f't={r["t"]:.2f}, p={r["p"]:.4f}')
else:
    print('  (Run Cell 17 first)')

# ── 5. Sub-008 influence ──
print('\n5. SUB-008 INFLUENCE')
print('-' * 95)
print('  Sub-008 is individually significant (Crawford-Howell) on:')
print('    Geometry preservation / word:  t=-3.35, p=.012')
print('    MDS shift / face, object, word')
print('    RDM distance / face, word')
print('    Liu distinctiveness / word')
print('  Removing sub-008 strengthens uni-bi effects (geometry, RDM)')
print('  but no group-level results cross significance threshold either way.')

# ── 6. Trending results (p < .10) ──
print('\n6. TRENDING (p < .10, not significant)')
print('-' * 95)
for _, r in rdf_sig.iterrows():
    for pcol, label in [('p OTC v Ctrl', 'OTC v Ctrl'),
                        ('p OTC v nonOTC', 'OTC v nonOTC'),
                        ('p nonOTC v Ctrl', 'nonOTC v Ctrl')]:
        p = r[pcol]
        if not np.isnan(p) and .05 <= p < .10:
            print(f'  {r["Category"]:<10} {r["Metric"]:<28} {label:<16} p={p:.3f}')

print('\n' + '=' * 95)

KEY RESULTS AT-A-GLANCE

1. GROUP-LEVEL BOOTSTRAP (OTC vs Controls, main table)
-----------------------------------------------------------------------------------------------
Category   Metric                             Ctrl M(SD)        OTC M(SD)       Comparison          p
-----------------------------------------------------------------------------------------------
word       Peak Drift (mm)                   5.76 (5.90)     11.47 (4.19)       OTC v Ctrl     0.022*
face       Mean Activation                   4.71 (1.29)      4.24 (1.18)     OTC v nonOTC     0.021*
house      Volume                       4868.92 (2890.45) 3421.31 (1877.43)       OTC v Ctrl     0.046*
object     Volume                       20816.00 (7370.01) 12242.12 (7229.33)       OTC v Ctrl     <.001*
object     Volume                       20816.00 (7370.01) 12242.12 (7229.33)    nonOTC v Ctrl     0.041*
object     Sum Selectivity              1942.04 (865.54) 1253.14 (1039.04)       OTC v Ctrl     0.025*
wor

In [20]:
# FDR Correction (Benjamini-Hochberg)
# ═══════════════════════════════════════════════════════════════

def benjamini_hochberg(pvals):
    """Return FDR-corrected p-values (Benjamini-Hochberg)."""
    pvals = np.asarray(pvals)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = np.empty(n)
    ranked[order] = np.arange(1, n + 1)
    adjusted = pvals * n / ranked
    # Enforce monotonicity (step-down)
    adjusted = np.minimum.accumulate(adjusted[np.argsort(ranked)[::-1]])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    # Restore original order
    result = np.empty(n)
    result[np.argsort(ranked).astype(int)] = adjusted
    return result

rdf_fdr = pd.DataFrame(results_rows)

# Collect all p-values with their source info
all_tests = []
for _, r in rdf_fdr.iterrows():
    for pcol, label in [('p OTC v Ctrl', 'OTC v Ctrl'),
                        ('p OTC v nonOTC', 'OTC v nonOTC'),
                        ('p nonOTC v Ctrl', 'nonOTC v Ctrl')]:
        p = r[pcol]
        if not np.isnan(p):
            all_tests.append({
                'Category': r['Category'], 'Metric': r['Metric'],
                'Comparison': label, 'p_uncorr': p,
            })

fdr_df = pd.DataFrame(all_tests)

# Full-table FDR
fdr_df['p_fdr_all'] = benjamini_hochberg(fdr_df['p_uncorr'].values)

# Within-metric FDR
fdr_df['p_fdr_metric'] = np.nan
for metric in fdr_df['Metric'].unique():
    mask = fdr_df['Metric'] == metric
    fdr_df.loc[mask, 'p_fdr_metric'] = benjamini_hochberg(
        fdr_df.loc[mask, 'p_uncorr'].values)

# Print results
print('=' * 110)
print('FDR CORRECTION (Benjamini-Hochberg)')
print('=' * 110)

sig_uncorr = fdr_df[fdr_df['p_uncorr'] < .05].sort_values('p_uncorr')
print(f'\n{"Category":<10} {"Metric":<28} {"Comparison":<16} '
      f'{"p(uncorr)":>10} {"p(FDR-metric)":>14} {"p(FDR-all)":>12}')
print('-' * 95)

for _, r in sig_uncorr.iterrows():
    def star(p): return '*' if p < .05 else ''
    print(f'{r["Category"]:<10} {r["Metric"]:<28} {r["Comparison"]:<16} '
          f'{r["p_uncorr"]:>10.4f}* '
          f'{r["p_fdr_metric"]:>13.4f}{star(r["p_fdr_metric"])} '
          f'{r["p_fdr_all"]:>11.4f}{star(r["p_fdr_all"])}')

print(f'\nTotal tests: {len(fdr_df)}')
print(f'Significant uncorrected (p<.05): {(fdr_df["p_uncorr"] < .05).sum()}')
print(f'Significant FDR within-metric:   {(fdr_df["p_fdr_metric"] < .05).sum()}')
print(f'Significant FDR full-table:      {(fdr_df["p_fdr_all"] < .05).sum()}')

FDR CORRECTION (Benjamini-Hochberg)

Category   Metric                       Comparison        p(uncorr)  p(FDR-metric)   p(FDR-all)
-----------------------------------------------------------------------------------------------
object     Volume                       OTC v Ctrl           0.0001*        0.0012*      0.0068*
face       Mean Activation              OTC v nonOTC         0.0210*        0.2520      0.3679
word       Peak Drift (mm)              OTC v Ctrl           0.0221*        0.0884      0.3679
object     Sum Selectivity              OTC v Ctrl           0.0254*        0.2094      0.3679
house      Delta Sum Selectivity        OTC v Ctrl           0.0287*        0.1148      0.3679
word       Sum Selectivity              nonOTC v Ctrl        0.0349*        0.2094      0.3679
object     Volume                       nonOTC v Ctrl        0.0405*        0.1856      0.3679
house      Volume                       OTC v Ctrl           0.0464*        0.1856      0.3679

Total te

In [21]:
# ═══════════════════════════════════════════════════════════════════════════════
# DIFF-OF-DIFF: Two control hemisphere approaches
#   1) Preferred functional: face=RH, word=LH, house=RH, object=LH
#   2) Anatomically matched: match patient's intact hemisphere
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.stats import ttest_rel
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path(processed_dir)
GEO_DIR = BASE / 'group_results' / 'geometry'
LIU_DIR = BASE / 'group_results' / 'liu_distinctiveness'
COPE_SET = 'differential'
EXCLUDE = ['sub-017']

CATEGORIES = ['face', 'house', 'object', 'word']
BILATERAL = ['house', 'object']
UNILATERAL = ['face', 'word']

PREFERRED_CTRL_HEMI = {
    'face': 'right', 'word': 'left', 'house': 'right', 'object': 'left',
}

N_BOOT = 100000
rng = np.random.default_rng(42)

# ── Load geometry ─────────────────────────────────────────────────────────
geo = pd.read_csv(GEO_DIR / f'geometry_{COPE_SET}.csv')
geo = geo[~geo['subject_id'].isin(EXCLUDE)]
geo = geo[geo['category'].isin(CATEGORIES)]

otc_geo = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]
ctrl_geo = geo[geo['status'] == 'control']

# ── Load pairwise for RDM distance ───────────────────────────────────────
pair_file = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pw = pd.read_csv(pair_file)
df_pw = df_pw[~df_pw['subject_id'].isin(EXCLUDE)]
df_pw = df_pw[df_pw['category'].isin(CATEGORIES)]

ALL_PAIRS = sorted(df_pw['pair'].unique())

# Longitudinal only
ses_c = df_pw.groupby('subject_id')['session'].nunique()
multi = ses_c[ses_c >= 2].index.tolist()
df_pw_long = df_pw[df_pw['subject_id'].isin(multi)].copy()
df_pw_long['ses_rank'] = df_pw_long.groupby('subject_id')['session'].rank(
    method='dense').astype(int)
max_rank = df_pw_long.groupby('subject_id')['ses_rank'].transform('max')
df_pw_long = df_pw_long[(df_pw_long['ses_rank'] == 1) |
                         (df_pw_long['ses_rank'] == max_rank)].copy()
df_pw_long['tp'] = df_pw_long['ses_rank'].apply(lambda x: 'T1' if x == 1 else 'T2')

def compute_rdm_distance(df, subject_id, roi_cat):
    sub_df = df[df['subject_id'] == subject_id]
    t1_vals, t2_vals = [], []
    for pair in ALL_PAIRS:
        t1 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T1')]['fisher_r']
        t2 = sub_df[(sub_df['category'] == roi_cat) &
                     (sub_df['pair'] == pair) &
                     (sub_df['tp'] == 'T2')]['fisher_r']
        if len(t1) > 0 and len(t2) > 0:
            t1_vals.append(t1.values[0])
            t2_vals.append(t2.values[0])
    if len(t1_vals) < 6:
        return np.nan
    return np.sqrt(np.sum((np.array(t1_vals) - np.array(t2_vals))**2))


def get_otc_uni_bi(df, value_col, metric_name):
    """Get per-patient unilateral and bilateral means for OTC."""
    otc = df[(df['group'] == 'OTC') & (df['hemi_label'] == 'intact')]
    otc = otc[otc['category'].isin(CATEGORIES)]
    uni_m, bi_m, subs = [], [], []
    for sub in sorted(otc['subject_id'].unique()):
        sd = otc[otc['subject_id'] == sub]
        u = sd[sd['category'].isin(UNILATERAL)][value_col].dropna().values
        b = sd[sd['category'].isin(BILATERAL)][value_col].dropna().values
        if len(u) > 0 and len(b) > 0:
            uni_m.append(np.mean(u))
            bi_m.append(np.mean(b))
            subs.append(sub)
    return np.array(uni_m), np.array(bi_m), subs


def get_ctrl_uni_bi_preferred(df, value_col):
    """Controls at preferred hemisphere per category."""
    rows = []
    for sub in df[df['status'] == 'control']['subject_id'].unique():
        sd = df[(df['subject_id'] == sub) & (df['status'] == 'control')]
        u_vals, b_vals = [], []
        for cat in UNILATERAL:
            pref = PREFERRED_CTRL_HEMI[cat]
            v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)][value_col].values
            if len(v) > 0:
                u_vals.append(v[0])
        for cat in BILATERAL:
            pref = PREFERRED_CTRL_HEMI[cat]
            v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)][value_col].values
            if len(v) > 0:
                b_vals.append(v[0])
        if len(u_vals) > 0 and len(b_vals) > 0:
            rows.append({'sub': sub, 'uni': np.mean(u_vals), 'bi': np.mean(b_vals)})
    cdf = pd.DataFrame(rows)
    return cdf['uni'].values, cdf['bi'].values


def get_ctrl_uni_bi_matched(df, value_col, hemi_label):
    """Controls at one specific hemisphere for all categories."""
    rows = []
    for sub in df[df['status'] == 'control']['subject_id'].unique():
        sd = df[(df['subject_id'] == sub) & (df['status'] == 'control') &
                (df['hemi_label'] == hemi_label)]
        u_vals = sd[sd['category'].isin(UNILATERAL)][value_col].dropna().values
        b_vals = sd[sd['category'].isin(BILATERAL)][value_col].dropna().values
        if len(u_vals) > 0 and len(b_vals) > 0:
            rows.append({'sub': sub, 'uni': np.mean(u_vals), 'bi': np.mean(b_vals)})
    cdf = pd.DataFrame(rows)
    if len(cdf) == 0:
        return np.array([]), np.array([])
    return cdf['uni'].values, cdf['bi'].values


def run_diff_of_diff(otc_uni, otc_bi, ctrl_uni, ctrl_bi, label):
    """Run paired test + bootstrap diff-of-diff."""
    otc_diff = otc_uni - otc_bi  # for geometry: positive = uni better
    ctrl_diff = ctrl_uni - ctrl_bi

    print(f'\n  {label}:')
    print(f'    OTC:  uni M={np.mean(otc_uni):.3f}, bi M={np.mean(otc_bi):.3f}, '
          f'diff M={np.mean(otc_diff):.3f} (n={len(otc_diff)})')
    print(f'    Ctrl: uni M={np.mean(ctrl_uni):.3f}, bi M={np.mean(ctrl_bi):.3f}, '
          f'diff M={np.mean(ctrl_diff):.3f} (n={len(ctrl_diff)})')

    dd = np.mean(otc_diff) - np.mean(ctrl_diff)
    boot = np.empty(N_BOOT)
    for i in range(N_BOOT):
        o = rng.choice(otc_diff, size=len(otc_diff), replace=True)
        c = rng.choice(ctrl_diff, size=len(ctrl_diff), replace=True)
        boot[i] = o.mean() - c.mean()
    ci = np.percentile(boot, [2.5, 97.5])
    p = 2 * min(np.mean(boot <= 0), np.mean(boot >= 0))
    p = min(p, 1.0)
    sig = '*' if p < .05 else ''

    print(f'    Diff-of-diff: {dd:+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}], p={p:.4f} {sig}')


# ══════════════════════════════════════════════════════════════════════════
# GEOMETRY PRESERVATION
# ══════════════════════════════════════════════════════════════════════════

print('='*70)
print('GEOMETRY PRESERVATION: Uni vs Bi diff-of-diff')
print('='*70)

otc_uni_g, otc_bi_g, subs_g = get_otc_uni_bi(geo, 'geometry_preservation', 'geo')

# Within OTC paired test
if len(otc_uni_g) >= 3:
    t, p = ttest_rel(otc_uni_g, otc_bi_g)
    print(f'\n  OTC within-patient: t({len(otc_uni_g)-1})={t:.3f}, p={p:.4f}')
    print(f'  Subjects: {subs_g}')

# Approach 1: Preferred functional hemisphere
ctrl_uni_pref, ctrl_bi_pref = get_ctrl_uni_bi_preferred(ctrl_geo, 'geometry_preservation')
run_diff_of_diff(otc_uni_g, otc_bi_g, ctrl_uni_pref, ctrl_bi_pref,
                 'Approach 1: Controls at PREFERRED hemisphere')

# Approach 2: Anatomically matched
# Split OTC by intact hemisphere, compare each to matched controls
for intact_hemi, hemi_label in [('left', 'left'), ('right', 'right')]:
    otc_hemi = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact') &
                    (geo['surgery_side'] != intact_hemi)]  # surgery opposite of intact
    # Actually just filter by intact hemisphere directly
    otc_hemi = geo[(geo['group'] == 'OTC') & (geo['hemi_label'] == 'intact')]
    # Get surgery_side to determine intact
    otc_intact = otc_hemi.copy()
    otc_intact['intact_hemi'] = otc_intact['surgery_side'].map(
        lambda s: 'right' if s == 'left' else 'left')
    otc_this = otc_intact[otc_intact['intact_hemi'] == intact_hemi]

    if len(otc_this['subject_id'].unique()) == 0:
        continue

    uni_m, bi_m, subs_h = [], [], []
    for sub in sorted(otc_this['subject_id'].unique()):
        sd = otc_this[otc_this['subject_id'] == sub]
        u = sd[sd['category'].isin(UNILATERAL)]['geometry_preservation'].dropna().values
        b = sd[sd['category'].isin(BILATERAL)]['geometry_preservation'].dropna().values
        if len(u) > 0 and len(b) > 0:
            uni_m.append(np.mean(u))
            bi_m.append(np.mean(b))
            subs_h.append(sub)

    ctrl_uni_m, ctrl_bi_m = get_ctrl_uni_bi_matched(
        ctrl_geo, 'geometry_preservation', hemi_label)

    if len(uni_m) > 0 and len(ctrl_uni_m) > 0:
        run_diff_of_diff(np.array(uni_m), np.array(bi_m),
                         ctrl_uni_m, ctrl_bi_m,
                         f'Approach 2: Controls at {hemi_label.upper()} (matched to intact {intact_hemi}), '
                         f'n_otc={len(uni_m)}, subs={subs_h}')

# ══════════════════════════════════════════════════════════════════════════
# RDM DISTANCE
# ══════════════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('RDM DISTANCE: Uni vs Bi diff-of-diff')
print('='*70)

# Compute RDM distance per OTC patient × category
otc_pw = df_pw_long[(df_pw_long['group'] == 'OTC') &
                     (df_pw_long['hemi_label'] == 'intact')]
ctrl_pw = df_pw_long[df_pw_long['status'] == 'control']

rdm_rows_otc = []
for sub in sorted(otc_pw['subject_id'].unique()):
    sub_code = otc_pw[otc_pw['subject_id'] == sub]['subject'].iloc[0]
    surgery = otc_pw[otc_pw['subject_id'] == sub]['surgery_side'].iloc[0]
    intact = 'right' if surgery == 'left' else 'left'
    for cat in CATEGORIES:
        d = compute_rdm_distance(otc_pw, sub, cat)
        if np.isfinite(d):
            rdm_rows_otc.append({
                'subject_id': sub, 'subject': sub_code,
                'intact_hemi': intact, 'category': cat,
                'cat_type': 'bilateral' if cat in BILATERAL else 'unilateral',
                'rdm_distance': d})

rdm_rows_ctrl = []
for sub in sorted(ctrl_pw['subject_id'].unique()):
    for hemi_label in ['left', 'right']:
        sub_hemi = ctrl_pw[(ctrl_pw['subject_id'] == sub) &
                            (ctrl_pw['hemi_label'] == hemi_label)]
        if len(sub_hemi) == 0:
            continue
        for cat in CATEGORIES:
            d = compute_rdm_distance(sub_hemi, sub, cat)
            if np.isfinite(d):
                rdm_rows_ctrl.append({
                    'subject_id': sub, 'hemi_label': hemi_label,
                    'category': cat,
                    'cat_type': 'bilateral' if cat in BILATERAL else 'unilateral',
                    'rdm_distance': d})

df_rdm_otc = pd.DataFrame(rdm_rows_otc)
df_rdm_ctrl = pd.DataFrame(rdm_rows_ctrl)

# OTC uni/bi
otc_rdm_uni, otc_rdm_bi, subs_r = [], [], []
for sub in df_rdm_otc['subject_id'].unique():
    sd = df_rdm_otc[df_rdm_otc['subject_id'] == sub]
    u = sd[sd['cat_type'] == 'unilateral']['rdm_distance'].mean()
    b = sd[sd['cat_type'] == 'bilateral']['rdm_distance'].mean()
    if np.isfinite(u) and np.isfinite(b):
        otc_rdm_uni.append(u)
        otc_rdm_bi.append(b)
        subs_r.append(sub)
otc_rdm_uni = np.array(otc_rdm_uni)
otc_rdm_bi = np.array(otc_rdm_bi)

if len(otc_rdm_uni) >= 3:
    t, p = ttest_rel(otc_rdm_uni, otc_rdm_bi)
    print(f'\n  OTC within-patient: t({len(otc_rdm_uni)-1})={t:.3f}, p={p:.4f}')
    print(f'  Subjects: {subs_r}')

# Approach 1: Preferred
ctrl_rdm_pref_rows = []
for sub in df_rdm_ctrl['subject_id'].unique():
    sd = df_rdm_ctrl[df_rdm_ctrl['subject_id'] == sub]
    u_vals, b_vals = [], []
    for cat in UNILATERAL:
        pref = PREFERRED_CTRL_HEMI[cat]
        v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)]['rdm_distance'].values
        if len(v) > 0:
            u_vals.append(v[0])
    for cat in BILATERAL:
        pref = PREFERRED_CTRL_HEMI[cat]
        v = sd[(sd['category'] == cat) & (sd['hemi_label'] == pref)]['rdm_distance'].values
        if len(v) > 0:
            b_vals.append(v[0])
    if len(u_vals) > 0 and len(b_vals) > 0:
        ctrl_rdm_pref_rows.append({'uni': np.mean(u_vals), 'bi': np.mean(b_vals)})

cdf_pref = pd.DataFrame(ctrl_rdm_pref_rows)
run_diff_of_diff(otc_rdm_uni, otc_rdm_bi,
                 cdf_pref['uni'].values, cdf_pref['bi'].values,
                 'Approach 1: Controls at PREFERRED hemisphere')

# Approach 2: Anatomically matched
for intact_hemi in ['left', 'right']:
    otc_this = df_rdm_otc[df_rdm_otc['intact_hemi'] == intact_hemi]
    if len(otc_this['subject_id'].unique()) == 0:
        continue

    uni_m, bi_m, subs_h = [], [], []
    for sub in sorted(otc_this['subject_id'].unique()):
        sd = otc_this[otc_this['subject_id'] == sub]
        u = sd[sd['cat_type'] == 'unilateral']['rdm_distance'].mean()
        b = sd[sd['cat_type'] == 'bilateral']['rdm_distance'].mean()
        if np.isfinite(u) and np.isfinite(b):
            uni_m.append(u)
            bi_m.append(b)
            subs_h.append(sub)

    ctrl_hemi = df_rdm_ctrl[df_rdm_ctrl['hemi_label'] == intact_hemi]
    ctrl_uni_h, ctrl_bi_h = [], []
    for sub in ctrl_hemi['subject_id'].unique():
        sd = ctrl_hemi[ctrl_hemi['subject_id'] == sub]
        u = sd[sd['cat_type'] == 'unilateral']['rdm_distance'].mean()
        b = sd[sd['cat_type'] == 'bilateral']['rdm_distance'].mean()
        if np.isfinite(u) and np.isfinite(b):
            ctrl_uni_h.append(u)
            ctrl_bi_h.append(b)

    if len(uni_m) > 0 and len(ctrl_uni_h) > 0:
        run_diff_of_diff(np.array(uni_m), np.array(bi_m),
                         np.array(ctrl_uni_h), np.array(ctrl_bi_h),
                         f'Approach 2: Controls at {intact_hemi.upper()} '
                         f'(matched), n_otc={len(uni_m)}, subs={subs_h}')

# ══════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('SUMMARY')
print('='*70)
print("""
Approach 1 (Preferred): Each control category uses its functionally
  preferred hemisphere (face=RH, word=LH, house=RH, object=LH).
  Controls get their "best" hemisphere per category.

Approach 2 (Anatomically matched): All control categories use the
  same hemisphere that the patient group has intact.
  L-resection patients (intact RH) → controls' RH for everything.
  R-resection patients (intact LH) → controls' LH for everything.
  This means word is compared against controls' RH (where word is weak).
""")

GEOMETRY PRESERVATION: Uni vs Bi diff-of-diff

  OTC within-patient: t(4)=4.378, p=0.0119
  Subjects: ['sub-004', 'sub-008', 'sub-010', 'sub-021', 'sub-079']

  Approach 1: Controls at PREFERRED hemisphere:
    OTC:  uni M=0.671, bi M=0.238, diff M=0.432 (n=5)
    Ctrl: uni M=0.714, bi M=0.465, diff M=0.249 (n=9)
    Diff-of-diff: +0.183, 95% CI [-0.050, +0.427], p=0.1278 

  Approach 2: Controls at LEFT (matched to intact left), n_otc=2, subs=['sub-004', 'sub-008']:
    OTC:  uni M=0.440, bi M=0.071, diff M=0.369 (n=2)
    Ctrl: uni M=0.702, bi M=0.509, diff M=0.193 (n=9)
    Diff-of-diff: +0.176, 95% CI [-0.184, +0.538], p=0.3495 

  Approach 2: Controls at RIGHT (matched to intact right), n_otc=3, subs=['sub-010', 'sub-021', 'sub-079']:
    OTC:  uni M=0.824, bi M=0.350, diff M=0.475 (n=3)
    Ctrl: uni M=0.486, bi M=0.518, diff M=-0.032 (n=9)
    Diff-of-diff: +0.507, 95% CI [+0.207, +0.831], p=0.0002 *

RDM DISTANCE: Uni vs Bi diff-of-diff

  OTC within-patient: t(4)=-2.155, p=0.0

In [23]:
# ═══════════════════════════════════════════════════════════════════════════════
# MASTER TABLE: Two control hemisphere approaches
# ═══════════════════════════════════════════════════════════════════════════════
# ── Load data if not already available ────────────────────────────────────
SEL_DIR = BASE / 'group_results' / 'selectivity'
sel_file = SEL_DIR / 'selectivity_summary.csv'
df_sel_all = pd.read_csv(sel_file)
df_sel_all = df_sel_all[~df_sel_all['sub'].isin(['sub-017'])]
df_sel_all['ses_int'] = df_sel_all['ses'].astype(int)
first = df_sel_all.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
df_sel_first = df_sel_all.merge(first, on='sub')
df_sel_first = df_sel_first[df_sel_first['ses_int'] == df_sel_first['fs']]

def fmt_msd(m, sd):
    if np.isnan(m): return chr(8212)
    if np.isnan(sd): return f'{m:.2f}'
    return f'{m:.2f} ({sd:.2f})'

def fmt_v(v):
    return chr(8212) if np.isnan(v) else f'{v:.2f}'

def fmt_p(p):
    if np.isnan(p): return chr(8212)
    if p < .001: return '<.001*'
    return f'{p:.3f}' + ('*' if p < .05 else '')

METRIC_ORDER = [
    'Peak Drift (mm)',
    'Mean Activation', 'Volume', 'Sum Selectivity',
    'Delta Sum Selectivity',
    'Liu Distinctiveness',
    'Delta Liu Distinctiveness',
    'Geometry Preservation',
    'MDS Shift',
    'RDM Distance',
]

# ═══════════════════════════════════════════════════════════════════════════
# APPROACH 1: Functionally Preferred Hemisphere
# (This is what the existing results_rows already contains)
# ═══════════════════════════════════════════════════════════════════════════

rdf1 = pd.DataFrame(results_rows)
cat_ord = {c: i for i, c in enumerate(CATEGORIES)}
met_ord = {m: i for i, m in enumerate(METRIC_ORDER)}
rdf1['_c'] = rdf1['Category'].map(cat_ord)
rdf1['_m'] = rdf1['Metric'].map(met_ord)
rdf1 = rdf1.sort_values(['_c', '_m']).drop(columns=['_c', '_m'])

rows1 = []
for _, r in rdf1.iterrows():
    rows1.append({
        'Category':      r['Category'].capitalize(),
        'Metric':        r['Metric'],
        'Ctrl M(SD)':    fmt_msd(r['Ctrl M'], r['Ctrl SD']),
        'OTC M(SD)':     fmt_msd(r['OTC M'], r['OTC SD']),
        'nonOTC M(SD)':  fmt_msd(r['nonOTC M'], r['nonOTC SD']),
        'OTC L-resec':   fmt_v(r['OTC L-resec']),
        'OTC R-resec':   fmt_v(r['OTC R-resec']),
        'OTC v Ctrl':    fmt_p(r['p OTC v Ctrl']),
        'OTC v nonOTC':  fmt_p(r['p OTC v nonOTC']),
        'nonOTC v Ctrl': fmt_p(r['p nonOTC v Ctrl']),
    })

display1 = pd.DataFrame(rows1)

print('='*120)
print('APPROACH 1: FUNCTIONALLY PREFERRED HEMISPHERE')
print('='*120)
print('Controls @ preferred hemi per category: face=RH, house=RH, object=LH, word=LH')
print('Patients @ intact hemisphere')
print('='*120)
print()
print(display1.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# APPROACH 2: Anatomically Matched Hemisphere
# Re-extract everything with controls matched to patient's intact side
# L-resection patients (intact RH) → controls RH
# R-resection patients (intact LH) → controls LH
# ═══════════════════════════════════════════════════════════════════════════

def extract_matched_geo(df, cat, value_col, ctrl_hemi, surgery_side, cross_sectional=False):
    """Extract values matching controls to a specific hemisphere."""
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['status'] == 'control') & (c['hemi_label'] == ctrl_hemi)][value_col].dropna().values
    otc = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact') &
            (c['surgery_side'] == surgery_side)]
    otc_vals = otc[value_col].dropna().values
    nonotc_vals = np.array([])
    if cross_sectional:
        nonotc = c[(c['group'] == 'nonOTC') & (c['hemi_label'] == 'intact') &
                   (c['surgery_side'] == surgery_side)]
        nonotc_vals = nonotc[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals

def extract_matched_sel(df, cat, value_col, ctrl_hemi, intact_hemi):
    """Extract selectivity values with controls at specific hemisphere."""
    c = df[df['category'] == cat]
    ctrl_vals = c[(c['group'] == 'control') & (c['hemi'] == ctrl_hemi)][value_col].dropna().values
    otc_all = c[c['group'] == 'OTC']
    otc = otc_all[(otc_all['hemi'] == otc_all['intact_hemi']) &
                   (otc_all['intact_hemi'] == intact_hemi)]
    otc_vals = otc[value_col].dropna().values
    non_all = c[c['group'] == 'nonOTC']
    non = non_all[(non_all['hemi'] == non_all['intact_hemi']) &
                   (non_all['intact_hemi'] == intact_hemi)]
    nonotc_vals = non[value_col].dropna().values
    return ctrl_vals, otc_vals, nonotc_vals

results_rows_matched = []

def add_row_matched(cat, metric, ctrl, otc, nonotc, p_oc, resec_label,
                    p_on=np.nan, p_nc=np.nan):
    def m(a): return float(np.nanmean(a)) if len(a) > 0 else np.nan
    def s(a): return float(np.nanstd(a, ddof=1)) if len(a) > 1 else np.nan
    results_rows_matched.append({
        'Category': cat, 'Metric': metric, 'Resection': resec_label,
        'Ctrl M': m(ctrl), 'Ctrl SD': s(ctrl),
        'OTC M': m(otc), 'OTC SD': s(otc),
        'nonOTC M': m(nonotc), 'nonOTC SD': s(nonotc),
        'p OTC v Ctrl': p_oc,
        'p OTC v nonOTC': p_on,
        'p nonOTC v Ctrl': p_nc,
    })

# ── Re-extract all measures with matched hemispheres ──────────────────────

# For each resection side: L resection (intact RH) vs controls RH
#                          R resection (intact LH) vs controls LH

for surgery_side, intact_hemi, ctrl_hemi, label in [
    ('left', 'right', 'right', 'L-resec (intact RH) vs Ctrl RH'),
    ('right', 'left', 'left', 'R-resec (intact LH) vs Ctrl LH'),
]:
    # Selectivity (cross-sectional)
    for metric_name, metric_col in [('Mean Activation', 'mean_act'),
                                     ('Volume', 'volume'),
                                     ('Sum Selectivity', 'sum_selec_norm')]:
        for cat in CATEGORIES:
            cv, ov, nv = extract_matched_sel(
                df_sel_first, cat, metric_col, ctrl_hemi, intact_hemi)
            p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
            p_nc = bootstrap_p(nv, cv) if len(nv) > 0 and len(cv) > 0 else np.nan
            p_on = bootstrap_p(ov, nv) if len(ov) > 0 and len(nv) > 0 else np.nan
            add_row_matched(cat, metric_name, cv, ov, nv, p_oc, label, p_on, p_nc)

    # Liu distinctiveness (cross-sectional)
    if 'df_liu' in dir() or 'df_liu' in globals():
        liu_cs = df_liu.copy() if 'df_liu' in dir() else globals()['df_liu'].copy()
        liu_cs = liu_cs[~liu_cs['subject_id'].isin(EXCLUDE)]
        liu_cs = liu_cs[liu_cs['category'].isin(CATEGORIES)]
        # First session
        if 'ses_num' not in liu_cs.columns:
            liu_cs['ses_num'] = liu_cs.groupby('subject_id')['session'].rank(
                method='dense').astype(int)
        liu_first = liu_cs[liu_cs['ses_num'] == 1]
        for cat in CATEGORIES:
            cv = liu_first[(liu_first['status'] == 'control') &
                           (liu_first['hemi_label'] == ctrl_hemi) &
                           (liu_first['category'] == cat)]['liu_distinctiveness'].dropna().values
            otc_cat = liu_first[(liu_first['group'] == 'OTC') &
                                (liu_first['hemi_label'] == 'intact') &
                                (liu_first['category'] == cat)]
            # Filter by surgery side
            ov = otc_cat[otc_cat['surgery_side'] == surgery_side]['liu_distinctiveness'].dropna().values
            p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
            add_row_matched(cat, 'Liu Distinctiveness', cv, ov, np.array([]), p_oc, label)

    # Geometry (longitudinal)
    for cat in CATEGORIES:
        c = df_geo[df_geo['category'] == cat]
        cv = c[(c['status'] == 'control') &
               (c['hemi_label'] == ctrl_hemi)]['geometry_preservation'].dropna().values
        otc_cat = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact') &
                     (c['surgery_side'] == surgery_side)]
        ov = otc_cat['geometry_preservation'].dropna().values
        p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
        add_row_matched(cat, 'Geometry Preservation', cv, ov, np.array([]), p_oc, label)

    # RDM Distance (longitudinal)
    for cat in CATEGORIES:
        cv = df_rdm_ctrl[df_rdm_ctrl['hemi_label'] == ctrl_hemi]
        cv = cv[cv['category'] == cat]['rdm_distance'].dropna().values
        ov = df_rdm_otc[(df_rdm_otc['intact_hemi'] == intact_hemi) &
                         (df_rdm_otc['category'] == cat)]['rdm_distance'].dropna().values
        p_oc = bootstrap_p(ov, cv) if len(ov) > 0 and len(cv) > 0 else np.nan
        add_row_matched(cat, 'RDM Distance', cv, ov, np.array([]), p_oc, label)

# Format approach 2
rdf2 = pd.DataFrame(results_rows_matched)
rdf2['_c'] = rdf2['Category'].map(cat_ord)
rdf2['_m'] = rdf2['Metric'].map(met_ord)
rdf2 = rdf2.sort_values(['Resection', '_c', '_m']).drop(columns=['_c', '_m'])

rows2 = []
for _, r in rdf2.iterrows():
    rows2.append({
        'Resection':     r['Resection'],
        'Category':      r['Category'].capitalize(),
        'Metric':        r['Metric'],
        'Ctrl M(SD)':    fmt_msd(r['Ctrl M'], r['Ctrl SD']),
        'OTC M(SD)':     fmt_msd(r['OTC M'], r['OTC SD']),
        'OTC v Ctrl':    fmt_p(r['p OTC v Ctrl']),
    })

display2 = pd.DataFrame(rows2)

print('\n\n')
print('='*120)
print('APPROACH 2: ANATOMICALLY MATCHED HEMISPHERE')
print('='*120)
print('L-resection patients (intact RH) compared to controls RH for ALL categories')
print('R-resection patients (intact LH) compared to controls LH for ALL categories')
print('='*120)

for resec_label in display2['Resection'].unique():
    print(f'\n--- {resec_label} ---')
    sub = display2[display2['Resection'] == resec_label].drop(columns=['Resection'])
    print(sub.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# Save both
# ═══════════════════════════════════════════════════════════════════════════

display1.to_csv('master_results_approach1_preferred.csv', index=False)
display2.to_csv('master_results_approach2_matched.csv', index=False)
print(f'\nSaved: master_results_approach1_preferred.csv')
print(f'Saved: master_results_approach2_matched.csv')

APPROACH 1: FUNCTIONALLY PREFERRED HEMISPHERE
Controls @ preferred hemi per category: face=RH, house=RH, object=LH, word=LH
Patients @ intact hemisphere

Category                    Metric         Ctrl M(SD)          OTC M(SD)       nonOTC M(SD) OTC L-resec OTC R-resec OTC v Ctrl OTC v nonOTC nonOTC v Ctrl
    Face           Peak Drift (mm)        4.50 (9.89)        3.92 (4.10)                  —        1.89        6.96      0.868            —             —
    Face           Mean Activation        4.71 (1.29)        4.24 (1.18)        5.08 (0.74)        4.61        3.87      0.223       0.021*         0.290
    Face                    Volume  1547.25 (1093.53)  1305.19 (1058.18)  2112.22 (1358.74)     1140.62     1469.75      0.478        0.102         0.242
    Face           Sum Selectivity    492.96 (429.66)    415.60 (338.26)    675.24 (434.40)      386.33      444.88      0.519        0.103         0.257
    Face     Delta Sum Selectivity    115.52 (179.07)    -61.33 (516.72)    

In [24]:
# Key Results At-a-Glance (Restructured)
# ═══════════════════════════════════════════════════════════════

print('=' * 100)
print('KEY RESULTS AT-A-GLANCE')
print('=' * 100)

print("""
═══════════════════════════════════════════════════════════════════════
A. LONGITUDINAL FINDINGS (n=5 OTC, 9 controls)
═══════════════════════════════════════════════════════════════════════

1. BILATERAL REPRESENTATIONAL DEGRADATION > UNILATERAL
───────────────────────────────────────────────────────────────────────
  Geometry preservation (uni vs bi within OTC):
    t(4) = 4.378, p = .012*   uni=0.671 > bi=0.238
    Bootstrap diff-of-diff vs controls: p = .126 †

  RDM distance (bi vs uni within OTC):
    t(4) = -2.155, p = .098   bi=1.866 > uni=1.317
    Bootstrap diff-of-diff vs controls: p = .320 †

  House delta sum selectivity (OTC vs Ctrl):
    Ctrl: +274 (296)  OTC: -125 (399)   p = .029*

  † Excl sub-008: geometry uni-bi p=.018*, diff-of-diff p=.049*;
    RDM distance uni-bi p=.010*, diff-of-diff p=.030*.
    Sub-008 (hemispherectomy) is the sole patient where unilateral
    exceeds bilateral on RDM distance. Leave-one-out identifies
    sub-008 as an outlier; all other patients show the expected
    bilateral > unilateral pattern.

2. UNILATERAL CATEGORIES SPATIALLY RELOCATE
───────────────────────────────────────────────────────────────────────
  Word peak drift (OTC vs Ctrl):
    Ctrl: 5.76 (5.90) mm  OTC: 11.47 (4.19) mm   p = .022*
    L-resec: 13.47 mm  R-resec: 8.47 mm
    No other category shows significant drift.

3. DISSOCIATION: DRIFT ≠ DEGRADATION
───────────────────────────────────────────────────────────────────────
  Spatial drift does not predict representational change:
    Drift ↔ Geometry preservation: rho = -.386, p = .156

4. PER-CATEGORY GROUP COMPARISONS (OTC vs Ctrl, bootstrap)
───────────────────────────────────────────────────────────────────────
  Geometry preservation:
    face .899  house .210  object .450  word .209
    Word falls below control 95% CI (0.2nd percentile)

  RDM distance:
    face .849  house .373  object .121  word .550

  Delta sum selectivity:
    face .419  house .029*  object .558  word .961

  Delta Liu distinctiveness:
    face .627  house .780  object .547  word .051

5. SIGNIFICANT INDIVIDUAL PATIENTS (Crawford-Howell)
───────────────────────────────────────────────────────────────────────
  sub-008:  Geom/word p=.012, RDM/face p=.022, RDM/word p=.028,
            Liu/word p=.033
  sub-021:  Liu/object p=.007, Liu/word p=.014
  sub-076:  Liu/word p=.012

═══════════════════════════════════════════════════════════════════════
B. CROSS-SECTIONAL SAMPLE CHARACTERIZATION
═══════════════════════════════════════════════════════════════════════
  (n=16 OTC, 9 nonOTC, 24 controls)

  Volume reduction (OTC vs Ctrl):
    object p < .001*   house p = .046*
    face n.s.          word n.s.

  Sum selectivity reduction (OTC vs Ctrl):
    object p = .025*
    All others n.s.

  Mean activation:
    OTC vs nonOTC face p = .021* (OTC lower)
    All OTC vs Ctrl n.s.

  Liu distinctiveness (OTC mean vs control 95% CI):
    house: OTC above CI (99.5th pctile)
    object: OTC above CI (99.0th pctile)
    word: OTC above CI (100th pctile)

  Mantel (RDM correlation):
    OTC-R vs Ctrl: r = .940, p = .040*
    OTC-L vs Ctrl: r = .420, p = .295

  nonOTC vs Ctrl:
    object volume p = .041*
    word sum selectivity p = .035*

═══════════════════════════════════════════════════════════════════════
""")

KEY RESULTS AT-A-GLANCE

═══════════════════════════════════════════════════════════════════════
A. LONGITUDINAL FINDINGS (n=5 OTC, 9 controls)
═══════════════════════════════════════════════════════════════════════

1. BILATERAL REPRESENTATIONAL DEGRADATION > UNILATERAL
───────────────────────────────────────────────────────────────────────
  Geometry preservation (uni vs bi within OTC):
    t(4) = 4.378, p = .012*   uni=0.671 > bi=0.238
    Bootstrap diff-of-diff vs controls: p = .126 †

  RDM distance (bi vs uni within OTC):
    t(4) = -2.155, p = .098   bi=1.866 > uni=1.317
    Bootstrap diff-of-diff vs controls: p = .320 †

  House delta sum selectivity (OTC vs Ctrl):
    Ctrl: +274 (296)  OTC: -125 (399)   p = .029*

  † Excl sub-008: geometry uni-bi p=.018*, diff-of-diff p=.049*;
    RDM distance uni-bi p=.010*, diff-of-diff p=.030*.
    Sub-008 (hemispherectomy) is the sole patient where unilateral
    exceeds bilateral on RDM distance. Leave-one-out identifies
    sub-008 as a